# Phase 2 — Duplicate-Aware Split and ATE/ASC Export

This notebook performs the controlled splitting and task-specific export of the finance ABSA dataset.

Goals of this phase:

1. load the canonical Phase 1 dataset;
2. isolate unresolved review cases from the model-ready pool;
3. perform duplicate-aware, headline-level splitting using `group_key`;
4. apply a fixed, reproducible split policy;
5. export split-specific canonical files;
6. generate ATE-ready BIO supervision files;
7. generate ASC-ready sentence-aspect classification files;
8. verify that there is no split leakage and that exported files are internally consistent.

This phase does NOT train models yet.
It prepares the final shared dataset interface for Model A (ATE), Model B (ASC), and later pipeline/joint integration.

In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint

from sklearn.model_selection import train_test_split

In [2]:
IN_CANONICAL_JSONL = Path("outputs_phase1/finance_absa_phase1_canonical.jsonl")
IN_REVIEW_QUEUE = Path("outputs_phase1/span_review_queue.csv")

OUT_DIR = Path("outputs_phase2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_MANIFEST_PATH = OUT_DIR / "split_manifest.json"
SPLIT_STATS_PATH = OUT_DIR / "split_stats_report.json"
PHASE2_CHECKLIST_PATH = OUT_DIR / "phase2_checklist.csv"

CANONICAL_TRAIN_JSONL = OUT_DIR / "train_canonical.jsonl"
CANONICAL_VAL_JSONL   = OUT_DIR / "val_canonical.jsonl"
CANONICAL_TEST_JSONL  = OUT_DIR / "test_canonical.jsonl"
CANONICAL_REVIEW_JSONL = OUT_DIR / "review_holdout_canonical.jsonl"

ATE_TRAIN_JSONL = OUT_DIR / "ate_train.jsonl"
ATE_VAL_JSONL   = OUT_DIR / "ate_val.jsonl"
ATE_TEST_JSONL  = OUT_DIR / "ate_test.jsonl"
ATE_ALIGNMENT_ISSUES_JSONL = OUT_DIR / "ate_alignment_issues.jsonl"

ASC_TRAIN_JSONL = OUT_DIR / "asc_train.jsonl"
ASC_VAL_JSONL   = OUT_DIR / "asc_val.jsonl"
ASC_TEST_JSONL  = OUT_DIR / "asc_test.jsonl"

REVIEW_SUMMARY_PATH = OUT_DIR / "review_holdout_summary.json"
LEAKAGE_REPORT_PATH = OUT_DIR / "leakage_report.json"
ATE_VALIDATION_PATH = OUT_DIR / "ate_validation_report.json"
ASC_VALIDATION_PATH = OUT_DIR / "asc_validation_report.json"


## Step 1. Load the Phase 1 canonical dataset

We use the JSONL canonical file exported in Phase 1 as the sole source of truth for splitting and task-specific export.

In [3]:
records = []
with open(IN_CANONICAL_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

print("Canonical headline-level rows:", df.shape)
df.head(3)

Canonical headline-level rows: (10753, 7)


,raw_id,doc_id,text,group_key,num_aspects,label_signature,aspects
0,1,sentfin_000001,SpiceJet to issue 6.4 crore warrants to promoters,spicejet to issue 64 crore warrants to promoters,1,neutral,"[{'term': 'SpiceJet', 'sentiment': 'neutral', ..."
1,2,sentfin_000002,MMTC Q2 net loss at Rs 10.4 crore,mmtc q2 net loss at rs 104 crore,1,neutral,"[{'term': 'MMTC', 'sentiment': 'neutral', 'cha..."
2,3,sentfin_000003,"Mid-cap funds can deliver more, stay put: Experts",midcap funds can deliver more stay put experts,1,positive,"[{'term': 'Mid-cap funds', 'sentiment': 'posit..."


In [4]:
if IN_REVIEW_QUEUE.exists():
    df_review_queue = pd.read_csv(IN_REVIEW_QUEUE)
    print("Review queue rows:", len(df_review_queue))
else:
    df_review_queue = pd.DataFrame()
    print("No review queue file found.")

Review queue rows: 116


## Step 2. Inspect the canonical headline-level structure

Each row should represent one headline, with a list of aspect objects under the `aspects` field.
We now summarize how many headlines are fully matched versus partially unresolved.

In [5]:
def count_matched_aspects(aspects):
    return sum(1 for a in aspects if a.get("match_status") == "matched")

def count_review_aspects(aspects):
    return sum(1 for a in aspects if a.get("match_status") != "matched")

df["num_matched_aspects"] = df["aspects"].apply(count_matched_aspects)
df["num_review_aspects"] = df["aspects"].apply(count_review_aspects)
df["all_aspects_matched"] = df["num_review_aspects"] == 0

summary = {
    "headline_rows": int(len(df)),
    "all_aspects_matched_rows": int(df["all_aspects_matched"].sum()),
    "rows_with_any_review_aspect": int((~df["all_aspects_matched"]).sum()),
    "total_review_aspects": int(df["num_review_aspects"].sum()),
}
pprint(summary)

{'all_aspects_matched_rows': 10638,
 'headline_rows': 10753,
 'rows_with_any_review_aspect': 115,
 'total_review_aspects': 116}


## Step 3. Define split eligibility policy

Policy used in this project:

- Headlines with **all aspects matched** are eligible for train/val/test splitting.
- Headlines containing **any unresolved review aspect** are excluded from model-ready splits and moved to a separate review holdout.

Rationale:
For ATE, missing aspect spans would corrupt BIO supervision.
For ASC, unresolved grounding would weaken sentence-aspect consistency.
Since the unresolved pool is very small, the safest strategy is to isolate it completely.

In [6]:
df_model_ready = df[df["all_aspects_matched"]].copy().reset_index(drop=True)
df_review_holdout = df[~df["all_aspects_matched"]].copy().reset_index(drop=True)

print("Model-ready headline rows:", len(df_model_ready))
print("Review-holdout headline rows:", len(df_review_holdout))

Model-ready headline rows: 10638
Review-holdout headline rows: 115


In [7]:
with open(CANONICAL_REVIEW_JSONL, "w", encoding="utf-8") as f:
    for rec in df_review_holdout.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

review_summary = {
    "review_holdout_headlines": int(len(df_review_holdout)),
    "review_holdout_aspects": int(df_review_holdout["num_review_aspects"].sum()) if len(df_review_holdout) > 0 else 0,
}

with open(REVIEW_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(review_summary, f, ensure_ascii=False, indent=2)

print("Saved review-holdout canonical file:", CANONICAL_REVIEW_JSONL)
print("Saved review summary:", REVIEW_SUMMARY_PATH)
pprint(review_summary)

Saved review-holdout canonical file: outputs_phase2\review_holdout_canonical.jsonl
Saved review summary: outputs_phase2\review_holdout_summary.json
{'review_holdout_aspects': 116, 'review_holdout_headlines': 115}


## Step 4. Build a group-level split table

We split by `group_key`, not by individual headline row.

This prevents duplicate or near-duplicate headlines from being placed into different splits.

In [8]:
group_rows = []

for group_key, group in df_model_ready.groupby("group_key"):
    sig_counts = group["label_signature"].value_counts()
    dominant_signature = sig_counts.idxmax()
    group_rows.append({
        "group_key": group_key,
        "group_size": len(group),
        "dominant_label_signature": dominant_signature,
        "num_unique_signatures_in_group": group["label_signature"].nunique(),
        "row_ids": list(group["raw_id"])
    })

df_groups = pd.DataFrame(group_rows)
print("Group-level rows:", len(df_groups))
df_groups.head()

Group-level rows: 10570


,group_key,group_size,dominant_label_signature,num_unique_signatures_in_group,row_ids
0,10 takeaways from the secondquarter results of...,1,neutral,1,[5672]
1,10 year bond yield drops to 15 month low ahead...,1,positive,1,[4530]
2,100 days of narendra modi government sensex ra...,1,positive,1,[8265]
3,100 days of vishal sikka infosys up at 1612 pe...,1,positive,1,[8016]
4,100 infosys shares for 95k in 1993 now worth 4...,1,neutral,1,[6440]


In [9]:
mixed_groups = df_groups[df_groups["num_unique_signatures_in_group"] > 1].copy()
print("Groups with mixed label signatures:", len(mixed_groups))
mixed_groups.head(10)

Groups with mixed label signatures: 10


,group_key,group_size,dominant_label_signature,num_unique_signatures_in_group,row_ids
612,asian shares flat as investors digest greek deal,2,neutral,2,"[5695, 8950]"
1573,carl icahn says his firm sold remainder of net...,2,neutral,2,"[6246, 9400]"
2847,empee distilleries reports q3 net loss at rs 2...,2,neutral,2,"[3960, 9428]"
3349,flat nifty oi data hints at trend reversal,2,neutral,2,"[5764, 9016]"
4338,huge activity in nifty december series surpris...,2,positive,2,"[1612, 7281]"
5236,kenneth andrade exits idfc mutual fund leaving...,2,neutral,2,"[6076, 9256]"
5919,midcaps important part of our portfolio kunj b...,2,neutral+positive,2,"[2626, 8152]"
6461,nifty50 in a notrade zone due to holidaytrunca...,2,neutral,2,"[1147, 4262]"
7385,psus languishing in valuations but offer pocke...,2,negative,2,"[3815, 6139]"
10405,with investors steering clear infra bonds by b...,2,negative,2,"[4528, 6764]"


## Step 5. Define the split policy

Default split policy used here:

- train: 80%
- val: 10%
- test: 10%

The split is:
- group-aware,
- reproducible,
- stratified by dominant headline-level `label_signature`.

In [10]:
RANDOM_SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

## Step 6. Perform group-aware stratified split

We first split group rows into train and temp,
then split temp into val and test using the same fixed seed.

In [11]:
df_groups_train, df_groups_temp = train_test_split(
    df_groups,
    test_size=(1.0 - TRAIN_RATIO),
    random_state=RANDOM_SEED,
    stratify=df_groups["dominant_label_signature"]
)

temp_ratio = VAL_RATIO + TEST_RATIO
val_relative_ratio = VAL_RATIO / temp_ratio

df_groups_val, df_groups_test = train_test_split(
    df_groups_temp,
    test_size=(TEST_RATIO / temp_ratio),
    random_state=RANDOM_SEED,
    stratify=df_groups_temp["dominant_label_signature"]
)

print("Train groups:", len(df_groups_train))
print("Val groups:  ", len(df_groups_val))
print("Test groups: ", len(df_groups_test))

Train groups: 8456
Val groups:   1057
Test groups:  1057


## Step 7. Assign split labels back to headline-level rows

In [12]:
group_to_split = {}

for g in df_groups_train["group_key"]:
    group_to_split[g] = "train"
for g in df_groups_val["group_key"]:
    group_to_split[g] = "val"
for g in df_groups_test["group_key"]:
    group_to_split[g] = "test"

df_model_ready["split"] = df_model_ready["group_key"].map(group_to_split)

print(df_model_ready["split"].value_counts())

split
train    8509
test     1066
val      1063
Name: count, dtype: int64


In [13]:
assert df_model_ready["split"].isna().sum() == 0, "Some model-ready rows were not assigned a split."

## Step 8. Validate split integrity

We verify:
1. no `group_key` appears in more than one split;
2. no `raw_id` appears in more than one split;
3. split assignment is complete.

In [13]:
split_to_groups = {
    "train": set(df_model_ready.loc[df_model_ready["split"] == "train", "group_key"]),
    "val":   set(df_model_ready.loc[df_model_ready["split"] == "val", "group_key"]),
    "test":  set(df_model_ready.loc[df_model_ready["split"] == "test", "group_key"]),
}

leakage_report = {
    "train_val_overlap": len(split_to_groups["train"] & split_to_groups["val"]),
    "train_test_overlap": len(split_to_groups["train"] & split_to_groups["test"]),
    "val_test_overlap": len(split_to_groups["val"] & split_to_groups["test"]),
}

pprint(leakage_report)

{'train_test_overlap': 0, 'train_val_overlap': 0, 'val_test_overlap': 0}


In [14]:
split_to_raw_ids = {
    "train": set(df_model_ready.loc[df_model_ready["split"] == "train", "raw_id"]),
    "val":   set(df_model_ready.loc[df_model_ready["split"] == "val", "raw_id"]),
    "test":  set(df_model_ready.loc[df_model_ready["split"] == "test", "raw_id"]),
}

assert len(split_to_raw_ids["train"] & split_to_raw_ids["val"]) == 0
assert len(split_to_raw_ids["train"] & split_to_raw_ids["test"]) == 0
assert len(split_to_raw_ids["val"] & split_to_raw_ids["test"]) == 0

In [15]:
with open(LEAKAGE_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(leakage_report, f, ensure_ascii=False, indent=2)

print("Saved leakage report:", LEAKAGE_REPORT_PATH)

Saved leakage report: outputs_phase2\leakage_report.json


## Step 9. Validate label-signature distribution across splits

This is not expected to be perfectly identical, but the overall distribution should remain broadly similar.

In [16]:
overall_sig = df_model_ready["label_signature"].value_counts(normalize=True).sort_index()
train_sig = df_model_ready.loc[df_model_ready["split"] == "train", "label_signature"].value_counts(normalize=True).sort_index()
val_sig = df_model_ready.loc[df_model_ready["split"] == "val", "label_signature"].value_counts(normalize=True).sort_index()
test_sig = df_model_ready.loc[df_model_ready["split"] == "test", "label_signature"].value_counts(normalize=True).sort_index()

dist_df = pd.concat(
    [overall_sig.rename("overall"), train_sig.rename("train"), val_sig.rename("val"), test_sig.rename("test")],
    axis=1
).fillna(0)

dist_df

,overall,train,val,test
label_signature,,,,
negative,0.252209,0.252439,0.251176,0.251407
negative+neutral,0.040045,0.039958,0.040452,0.040338
negative+neutral+positive,0.001128,0.001058,0.001881,0.000938
negative+positive,0.013442,0.013515,0.013170,0.013133
neutral,0.319985,0.319897,0.318909,0.321764
neutral+positive,0.059880,0.059937,0.060207,0.059099
positive,0.313311,0.313198,0.314205,0.313321


In [17]:
split_stats = {
    "num_model_ready_rows": int(len(df_model_ready)),
    "num_train_rows": int((df_model_ready["split"] == "train").sum()),
    "num_val_rows": int((df_model_ready["split"] == "val").sum()),
    "num_test_rows": int((df_model_ready["split"] == "test").sum()),
    "num_train_groups": int(len(df_groups_train)),
    "num_val_groups": int(len(df_groups_val)),
    "num_test_groups": int(len(df_groups_test)),
    "review_holdout_rows": int(len(df_review_holdout)),
}
pprint(split_stats)

{'num_model_ready_rows': 10638,
 'num_test_groups': 1057,
 'num_test_rows': 1066,
 'num_train_groups': 8456,
 'num_train_rows': 8509,
 'num_val_groups': 1057,
 'num_val_rows': 1063,
 'review_holdout_rows': 115}


## Step 10. Export split-specific canonical files

These files remain headline-level and preserve the full list of matched aspects for each headline.
They are the shared interface for all downstream task-specific exports.

In [18]:
def save_jsonl_from_df(df_in, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in df_in.to_dict(orient="records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl_from_df(df_model_ready[df_model_ready["split"] == "train"], CANONICAL_TRAIN_JSONL)
save_jsonl_from_df(df_model_ready[df_model_ready["split"] == "val"],   CANONICAL_VAL_JSONL)
save_jsonl_from_df(df_model_ready[df_model_ready["split"] == "test"],  CANONICAL_TEST_JSONL)

print("Saved:", CANONICAL_TRAIN_JSONL)
print("Saved:", CANONICAL_VAL_JSONL)
print("Saved:", CANONICAL_TEST_JSONL)

Saved: outputs_phase2\train_canonical.jsonl
Saved: outputs_phase2\val_canonical.jsonl
Saved: outputs_phase2\test_canonical.jsonl


## Step 11. Build ATE-ready word-level supervision

ATE requires BIO sequence labeling.
We therefore convert each matched headline into:
- text
- token list
- token character offsets
- BIO labels
- aspect span metadata

Later, the model engineer can map these word-level BIO labels into DeBERTa subword labels.

In [19]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]")

def tokenize_with_offsets(text):
    return [(m.group(), m.start(), m.end()) for m in TOKEN_PATTERN.finditer(text)]

In [20]:
VALID_BIO_TAGS = {"O", "B-ASP", "I-ASP"}

def spans_overlap(a_from, a_to, b_from, b_to):
    return max(a_from, b_from) < min(a_to, b_to)

def build_bio_labels(tokens_with_offsets, aspects, text):
    labels = ["O"] * len(tokens_with_offsets)
    aspect_alignment_audit = []

    matched_aspects = [a for a in aspects if a.get("match_status") == "matched"]
    matched_aspects = sorted(matched_aspects, key=lambda x: (x["char_from"], x["char_to"]))

    for asp in matched_aspects:
        a_from = asp["char_from"]
        a_to = asp["char_to"]
        matched_text = text[a_from:a_to]

        overlap_token_indices = []
        full_token_indices = []

        for i, (_, t_from, t_to) in enumerate(tokens_with_offsets):
            if spans_overlap(t_from, t_to, a_from, a_to):
                overlap_token_indices.append(i)
                if t_from >= a_from and t_to <= a_to:
                    full_token_indices.append(i)

        if len(overlap_token_indices) == 0:
            coverage_status = "zero"
            token_indices = []
        else:
            overlap_start = min(tokens_with_offsets[i][1] for i in overlap_token_indices)
            overlap_end = max(tokens_with_offsets[i][2] for i in overlap_token_indices)
            clean_full = (
                len(full_token_indices) == len(overlap_token_indices)
                and overlap_start == a_from
                and overlap_end == a_to
            )
            coverage_status = "full" if clean_full else "partial"
            token_indices = full_token_indices if clean_full else overlap_token_indices

        if coverage_status == "full" and len(full_token_indices) > 0:
            labels[full_token_indices[0]] = "B-ASP"
            for j in full_token_indices[1:]:
                labels[j] = "I-ASP"

        aspect_alignment_audit.append({
            "term": asp["term"],
            "sentiment": asp["sentiment"],
            "char_from": a_from,
            "char_to": a_to,
            "match_type": asp["match_type"],
            "matched_text": matched_text,
            "token_indices": token_indices,
            "num_token_indices": len(token_indices),
            "coverage_status": coverage_status
        })

    return {
        "labels": labels,
        "aspect_alignment_audit": aspect_alignment_audit
    }


In [21]:
def make_ate_records(df_split):
    out = []
    for _, row in df_split.iterrows():
        text = row["text"]
        aspects = row["aspects"]
        toks = tokenize_with_offsets(text)
        bio_result = build_bio_labels(toks, aspects, text)
        bio = bio_result["labels"]
        aspect_alignment_audit = bio_result["aspect_alignment_audit"]

        num_matched_aspects = len(aspect_alignment_audit)
        num_full_coverage_aspects = sum(a["coverage_status"] == "full" for a in aspect_alignment_audit)
        num_partial_coverage_aspects = sum(a["coverage_status"] == "partial" for a in aspect_alignment_audit)
        num_zero_coverage_aspects = sum(a["coverage_status"] == "zero" for a in aspect_alignment_audit)
        has_all_o_but_matched_aspect = num_matched_aspects > 0 and all(tag == "O" for tag in bio)

        out.append({
            "doc_id": row["doc_id"],
            "raw_id": row["raw_id"],
            "split": row["split"],
            "text": text,
            "tokens": [t[0] for t in toks],
            "token_offsets": [(t[1], t[2]) for t in toks],
            "bio_labels": bio,
            "aspects": aspects,
            "num_matched_aspects": num_matched_aspects,
            "num_full_coverage_aspects": num_full_coverage_aspects,
            "num_partial_coverage_aspects": num_partial_coverage_aspects,
            "num_zero_coverage_aspects": num_zero_coverage_aspects,
            "has_all_o_but_matched_aspect": has_all_o_but_matched_aspect,
            "aspect_alignment_audit": aspect_alignment_audit
        })
    return out


## Step 12. Validate ATE supervision

We verify:
1. token count equals BIO label count;
2. no split leakage in exported ATE records;
3. BIO spans can be approximately recovered back into aspect surface forms.

In [22]:
ate_train = make_ate_records(df_model_ready[df_model_ready["split"] == "train"])
ate_val   = make_ate_records(df_model_ready[df_model_ready["split"] == "val"])
ate_test  = make_ate_records(df_model_ready[df_model_ready["split"] == "test"])

print(len(ate_train), len(ate_val), len(ate_test))

8509 1063 1066


In [23]:
def recover_bio_spans(tokens, labels):
    spans = []
    current = []
    for tok, lab in zip(tokens, labels):
        if lab == "B-ASP":
            if current:
                spans.append(" ".join(current))
            current = [tok]
        elif lab == "I-ASP":
            if current:
                current.append(tok)
        else:
            if current:
                spans.append(" ".join(current))
                current = []
    if current:
        spans.append(" ".join(current))
    return spans

def labels_are_valid(labels):
    return all(lab in VALID_BIO_TAGS for lab in labels)

def no_illegal_i_without_b(labels):
    inside_span = False
    for lab in labels:
        if lab == "B-ASP":
            inside_span = True
        elif lab == "I-ASP":
            if not inside_span:
                return False
        else:
            inside_span = False
    return True

def validate_ate_split(records):
    rows = len(records)
    token_label_len_ok = all(len(r["tokens"]) == len(r["bio_labels"]) == len(r["token_offsets"]) for r in records)
    bio_tags_valid = all(labels_are_valid(r["bio_labels"]) for r in records)
    no_illegal_i = all(no_illegal_i_without_b(r["bio_labels"]) for r in records)

    num_records_with_matched_aspects = 0
    num_records_all_o_but_has_matched_aspect = 0
    num_zero_coverage_aspects = 0
    num_partial_coverage_aspects = 0
    num_full_coverage_aspects = 0
    num_records_with_zero_coverage = 0
    num_records_with_partial_coverage = 0
    num_records_bio_span_count_mismatch = 0

    for rec in records:
        if rec["num_matched_aspects"] > 0:
            num_records_with_matched_aspects += 1
        if rec["has_all_o_but_matched_aspect"]:
            num_records_all_o_but_has_matched_aspect += 1

        num_zero_coverage_aspects += rec["num_zero_coverage_aspects"]
        num_partial_coverage_aspects += rec["num_partial_coverage_aspects"]
        num_full_coverage_aspects += rec["num_full_coverage_aspects"]

        if rec["num_zero_coverage_aspects"] > 0:
            num_records_with_zero_coverage += 1
        if rec["num_partial_coverage_aspects"] > 0:
            num_records_with_partial_coverage += 1

        recovered_bio_spans = recover_bio_spans(rec["tokens"], rec["bio_labels"])
        rec["num_bio_recovered_spans"] = len(recovered_bio_spans)
        rec["bio_span_count_mismatch"] = len(recovered_bio_spans) != rec["num_full_coverage_aspects"]
        if rec["bio_span_count_mismatch"]:
            num_records_bio_span_count_mismatch += 1

    return {
        "rows": rows,
        "token_label_len_ok": token_label_len_ok,
        "bio_tags_valid": bio_tags_valid,
        "no_illegal_i_without_b": no_illegal_i,
        "num_records_with_matched_aspects": num_records_with_matched_aspects,
        "num_records_all_o_but_has_matched_aspect": num_records_all_o_but_has_matched_aspect,
        "num_zero_coverage_aspects": num_zero_coverage_aspects,
        "num_partial_coverage_aspects": num_partial_coverage_aspects,
        "num_full_coverage_aspects": num_full_coverage_aspects,
        "num_records_with_zero_coverage": num_records_with_zero_coverage,
        "num_records_with_partial_coverage": num_records_with_partial_coverage,
        "num_records_bio_span_count_mismatch": num_records_bio_span_count_mismatch
    }

train_ate_validation = validate_ate_split(ate_train)
val_ate_validation = validate_ate_split(ate_val)
test_ate_validation = validate_ate_split(ate_test)

ate_validation = {
    "train_token_label_len_ok": train_ate_validation["token_label_len_ok"],
    "val_token_label_len_ok": val_ate_validation["token_label_len_ok"],
    "test_token_label_len_ok": test_ate_validation["token_label_len_ok"],
    "train_rows": len(ate_train),
    "val_rows": len(ate_val),
    "test_rows": len(ate_test),
    "train_bio_tags_valid": train_ate_validation["bio_tags_valid"],
    "val_bio_tags_valid": val_ate_validation["bio_tags_valid"],
    "test_bio_tags_valid": test_ate_validation["bio_tags_valid"],
    "train_no_illegal_i_without_b": train_ate_validation["no_illegal_i_without_b"],
    "val_no_illegal_i_without_b": val_ate_validation["no_illegal_i_without_b"],
    "test_no_illegal_i_without_b": test_ate_validation["no_illegal_i_without_b"],
    "train_num_records_with_matched_aspects": train_ate_validation["num_records_with_matched_aspects"],
    "val_num_records_with_matched_aspects": val_ate_validation["num_records_with_matched_aspects"],
    "test_num_records_with_matched_aspects": test_ate_validation["num_records_with_matched_aspects"],
    "train_num_records_all_o_but_has_matched_aspect": train_ate_validation["num_records_all_o_but_has_matched_aspect"],
    "val_num_records_all_o_but_has_matched_aspect": val_ate_validation["num_records_all_o_but_has_matched_aspect"],
    "test_num_records_all_o_but_has_matched_aspect": test_ate_validation["num_records_all_o_but_has_matched_aspect"],
    "train_num_zero_coverage_aspects": train_ate_validation["num_zero_coverage_aspects"],
    "val_num_zero_coverage_aspects": val_ate_validation["num_zero_coverage_aspects"],
    "test_num_zero_coverage_aspects": test_ate_validation["num_zero_coverage_aspects"],
    "train_num_partial_coverage_aspects": train_ate_validation["num_partial_coverage_aspects"],
    "val_num_partial_coverage_aspects": val_ate_validation["num_partial_coverage_aspects"],
    "test_num_partial_coverage_aspects": test_ate_validation["num_partial_coverage_aspects"],
    "train_num_full_coverage_aspects": train_ate_validation["num_full_coverage_aspects"],
    "val_num_full_coverage_aspects": val_ate_validation["num_full_coverage_aspects"],
    "test_num_full_coverage_aspects": test_ate_validation["num_full_coverage_aspects"],
    "train_num_records_with_zero_coverage": train_ate_validation["num_records_with_zero_coverage"],
    "val_num_records_with_zero_coverage": val_ate_validation["num_records_with_zero_coverage"],
    "test_num_records_with_zero_coverage": test_ate_validation["num_records_with_zero_coverage"],
    "train_num_records_with_partial_coverage": train_ate_validation["num_records_with_partial_coverage"],
    "val_num_records_with_partial_coverage": val_ate_validation["num_records_with_partial_coverage"],
    "test_num_records_with_partial_coverage": test_ate_validation["num_records_with_partial_coverage"],
    "train_num_records_bio_span_count_mismatch": train_ate_validation["num_records_bio_span_count_mismatch"],
    "val_num_records_bio_span_count_mismatch": val_ate_validation["num_records_bio_span_count_mismatch"],
    "test_num_records_bio_span_count_mismatch": test_ate_validation["num_records_bio_span_count_mismatch"]
}

pprint(ate_validation)


{'test_bio_tags_valid': True,
 'test_no_illegal_i_without_b': True,
 'test_num_full_coverage_aspects': 1445,
 'test_num_partial_coverage_aspects': 0,
 'test_num_records_all_o_but_has_matched_aspect': 0,
 'test_num_records_bio_span_count_mismatch': 0,
 'test_num_records_with_matched_aspects': 1066,
 'test_num_records_with_partial_coverage': 0,
 'test_num_records_with_zero_coverage': 0,
 'test_num_zero_coverage_aspects': 0,
 'test_rows': 1066,
 'test_token_label_len_ok': True,
 'train_bio_tags_valid': True,
 'train_no_illegal_i_without_b': True,
 'train_num_full_coverage_aspects': 11369,
 'train_num_partial_coverage_aspects': 0,
 'train_num_records_all_o_but_has_matched_aspect': 0,
 'train_num_records_bio_span_count_mismatch': 0,
 'train_num_records_with_matched_aspects': 8509,
 'train_num_records_with_partial_coverage': 0,
 'train_num_records_with_zero_coverage': 0,
 'train_num_zero_coverage_aspects': 0,
 'train_rows': 8509,
 'train_token_label_len_ok': True,
 'val_bio_tags_valid': True

In [24]:
with open(ATE_VALIDATION_PATH, "w", encoding="utf-8") as f:
    json.dump(ate_validation, f, ensure_ascii=False, indent=2)

print("Saved:", ATE_VALIDATION_PATH)


Saved: outputs_phase2\ate_validation_report.json


In [25]:
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(ate_train, ATE_TRAIN_JSONL)
save_jsonl(ate_val, ATE_VAL_JSONL)
save_jsonl(ate_test, ATE_TEST_JSONL)

ate_alignment_issues = []
for rec in ate_train + ate_val + ate_test:
    if (
        rec["has_all_o_but_matched_aspect"]
        or rec["num_zero_coverage_aspects"] > 0
        or rec["num_partial_coverage_aspects"] > 0
        or rec.get("bio_span_count_mismatch", False)
    ):
        ate_alignment_issues.append(rec)

save_jsonl(ate_alignment_issues, ATE_ALIGNMENT_ISSUES_JSONL)

print("Saved:", ATE_TRAIN_JSONL)
print("Saved:", ATE_VAL_JSONL)
print("Saved:", ATE_TEST_JSONL)
print("Saved:", ATE_ALIGNMENT_ISSUES_JSONL)
print("ATE alignment issue rows:", len(ate_alignment_issues))


Saved: outputs_phase2\ate_train.jsonl
Saved: outputs_phase2\ate_val.jsonl
Saved: outputs_phase2\ate_test.jsonl
Saved: outputs_phase2\ate_alignment_issues.jsonl
ATE alignment issue rows: 0


## Step 13. Build ASC-ready aspect-level classification records

ASC requires one record per (sentence, aspect, sentiment) pair.
Only matched aspects are exported.

In [26]:
def make_asc_records(df_split):
    out = []
    for _, row in df_split.iterrows():
        for asp in row["aspects"]:
            if asp.get("match_status") != "matched":
                continue
            out.append({
                "doc_id": row["doc_id"],
                "raw_id": row["raw_id"],
                "split": row["split"],
                "sentence": row["text"],
                "aspect": asp["term"],
                "sentiment": asp["sentiment"],
                "char_from": asp["char_from"],
                "char_to": asp["char_to"],
                "match_type": asp["match_type"],
                "input_text_template": f"{row['text']} [SEP] {asp['term']}"
            })
    return out

In [27]:
asc_train = make_asc_records(df_model_ready[df_model_ready["split"] == "train"])
asc_val   = make_asc_records(df_model_ready[df_model_ready["split"] == "val"])
asc_test  = make_asc_records(df_model_ready[df_model_ready["split"] == "test"])

print(len(asc_train), len(asc_val), len(asc_test))

11369 1417 1445


## Step 14. Validate ASC supervision

We verify:
1. no unresolved review aspects appear in ASC files;
2. all labels belong to the allowed 3-class set;
3. split counts are non-empty and consistent.

In [28]:
ALLOWED_SENTIMENTS = {"positive", "negative", "neutral"}
ALLOWED_ASC_MATCH_TYPES = {"whole_word_unique", "substring_unique", "normalized_unique"}

def validate_asc_split(records):
    all_labels_valid = True
    records_with_bad_span = 0
    records_with_bad_template = 0
    records_with_invalid_match_type = 0
    records_with_invalid_sentiment = 0

    for rec in records:
        if rec["sentiment"] not in ALLOWED_SENTIMENTS:
            all_labels_valid = False
            records_with_invalid_sentiment += 1

        if rec["match_type"] not in ALLOWED_ASC_MATCH_TYPES:
            records_with_invalid_match_type += 1

        if not (isinstance(rec["char_from"], int) and isinstance(rec["char_to"], int) and rec["char_from"] < rec["char_to"] and rec["sentence"][rec["char_from"]:rec["char_to"]] == rec["aspect"]):
            records_with_bad_span += 1

        expected_template = f"{rec['sentence']} [SEP] {rec['aspect']}"
        if rec["input_text_template"] != expected_template:
            records_with_bad_template += 1

    return {
        "records": len(records),
        "all_labels_valid": all_labels_valid,
        "records_with_bad_span": records_with_bad_span,
        "records_with_bad_template": records_with_bad_template,
        "records_with_invalid_match_type": records_with_invalid_match_type,
        "records_with_invalid_sentiment": records_with_invalid_sentiment
    }

train_asc_validation = validate_asc_split(asc_train)
val_asc_validation = validate_asc_split(asc_val)
test_asc_validation = validate_asc_split(asc_test)

asc_validation = {
    "train_all_labels_valid": train_asc_validation["all_labels_valid"],
    "val_all_labels_valid": val_asc_validation["all_labels_valid"],
    "test_all_labels_valid": test_asc_validation["all_labels_valid"],
    "train_records": len(asc_train),
    "val_records": len(asc_val),
    "test_records": len(asc_test),
    "train_records_with_bad_span": train_asc_validation["records_with_bad_span"],
    "val_records_with_bad_span": val_asc_validation["records_with_bad_span"],
    "test_records_with_bad_span": test_asc_validation["records_with_bad_span"],
    "train_records_with_bad_template": train_asc_validation["records_with_bad_template"],
    "val_records_with_bad_template": val_asc_validation["records_with_bad_template"],
    "test_records_with_bad_template": test_asc_validation["records_with_bad_template"],
    "train_records_with_invalid_match_type": train_asc_validation["records_with_invalid_match_type"],
    "val_records_with_invalid_match_type": val_asc_validation["records_with_invalid_match_type"],
    "test_records_with_invalid_match_type": test_asc_validation["records_with_invalid_match_type"],
    "train_records_with_invalid_sentiment": train_asc_validation["records_with_invalid_sentiment"],
    "val_records_with_invalid_sentiment": val_asc_validation["records_with_invalid_sentiment"],
    "test_records_with_invalid_sentiment": test_asc_validation["records_with_invalid_sentiment"]
}

pprint(asc_validation)


{'test_all_labels_valid': True,
 'test_records': 1445,
 'test_records_with_bad_span': 0,
 'test_records_with_bad_template': 0,
 'test_records_with_invalid_match_type': 0,
 'test_records_with_invalid_sentiment': 0,
 'train_all_labels_valid': True,
 'train_records': 11369,
 'train_records_with_bad_span': 0,
 'train_records_with_bad_template': 0,
 'train_records_with_invalid_match_type': 0,
 'train_records_with_invalid_sentiment': 0,
 'val_all_labels_valid': True,
 'val_records': 1417,
 'val_records_with_bad_span': 0,
 'val_records_with_bad_template': 0,
 'val_records_with_invalid_match_type': 0,
 'val_records_with_invalid_sentiment': 0}


In [29]:
with open(ASC_VALIDATION_PATH, "w", encoding="utf-8") as f:
    json.dump(asc_validation, f, ensure_ascii=False, indent=2)

print("Saved:", ASC_VALIDATION_PATH)

Saved: outputs_phase2\asc_validation_report.json


In [30]:
save_jsonl(asc_train, ASC_TRAIN_JSONL)
save_jsonl(asc_val, ASC_VAL_JSONL)
save_jsonl(asc_test, ASC_TEST_JSONL)

print("Saved:", ASC_TRAIN_JSONL)
print("Saved:", ASC_VAL_JSONL)
print("Saved:", ASC_TEST_JSONL)

Saved: outputs_phase2\asc_train.jsonl
Saved: outputs_phase2\asc_val.jsonl
Saved: outputs_phase2\asc_test.jsonl


## Step 15. Save the split manifest

This file records the exact split policy and key metadata required for reproducibility.

In [31]:
split_manifest = {
    "random_seed": RANDOM_SEED,
    "split_policy": {
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO
    },
    "split_unit": "group_key",
    "stratify_field": "dominant_label_signature",
    "review_policy": "headlines with any unresolved aspect are excluded to review holdout",
    "train_doc_ids": list(df_model_ready.loc[df_model_ready["split"] == "train", "doc_id"]),
    "val_doc_ids": list(df_model_ready.loc[df_model_ready["split"] == "val", "doc_id"]),
    "test_doc_ids": list(df_model_ready.loc[df_model_ready["split"] == "test", "doc_id"]),
}

with open(SPLIT_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, ensure_ascii=False, indent=2)

print("Saved split manifest:", SPLIT_MANIFEST_PATH)

Saved split manifest: outputs_phase2\split_manifest.json


## Step 16. Save split statistics report

In [32]:
split_stats_report = {
    **split_stats,
    "leakage_report": leakage_report,
    "review_summary": review_summary,
    "ate_validation": ate_validation,
    "asc_validation": asc_validation
}

with open(SPLIT_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(split_stats_report, f, ensure_ascii=False, indent=2)

print("Saved split stats report:", SPLIT_STATS_PATH)
pprint(split_stats_report)

Saved split stats report: outputs_phase2\split_stats_report.json
{'asc_validation': {'test_all_labels_valid': True,
                    'test_records': 1445,
                    'test_records_with_bad_span': 0,
                    'test_records_with_bad_template': 0,
                    'test_records_with_invalid_match_type': 0,
                    'test_records_with_invalid_sentiment': 0,
                    'train_all_labels_valid': True,
                    'train_records': 11369,
                    'train_records_with_bad_span': 0,
                    'train_records_with_bad_template': 0,
                    'train_records_with_invalid_match_type': 0,
                    'train_records_with_invalid_sentiment': 0,
                    'val_all_labels_valid': True,
                    'val_records': 1417,
                    'val_records_with_bad_span': 0,
                    'val_records_with_bad_template': 0,
                    'val_records_with_invalid_match_type': 0,
           

## Step 17. Evaluate whether Phase 2 is complete

Phase 2 is complete if:
- the model-ready pool is separated from the review holdout;
- duplicate-aware group splitting is performed;
- no group leakage exists across train/val/test;
- canonical split files are exported;
- ATE files are exported and BIO labels are well-formed;
- ASC files are exported and labels are valid;
- a split manifest is saved for reproducibility.

In [33]:
checklist = [
    ("Canonical input loaded", len(df) > 0),
    ("Review holdout created", CANONICAL_REVIEW_JSONL.exists()),
    ("All model-ready rows assigned split", df_model_ready["split"].isna().sum() == 0),
    ("No train/val group leakage", leakage_report["train_val_overlap"] == 0),
    ("No train/test group leakage", leakage_report["train_test_overlap"] == 0),
    ("No val/test group leakage", leakage_report["val_test_overlap"] == 0),
    ("Train canonical exported", CANONICAL_TRAIN_JSONL.exists()),
    ("Val canonical exported", CANONICAL_VAL_JSONL.exists()),
    ("Test canonical exported", CANONICAL_TEST_JSONL.exists()),
    ("ATE train exported", ATE_TRAIN_JSONL.exists()),
    ("ATE val exported", ATE_VAL_JSONL.exists()),
    ("ATE test exported", ATE_TEST_JSONL.exists()),
    ("ATE alignment issues exported", ATE_ALIGNMENT_ISSUES_JSONL.exists()),
    ("ASC train exported", ASC_TRAIN_JSONL.exists()),
    ("ASC val exported", ASC_VAL_JSONL.exists()),
    ("ASC test exported", ASC_TEST_JSONL.exists()),
    ("ATE validation passed", (
        ate_validation["train_token_label_len_ok"] and ate_validation["val_token_label_len_ok"] and ate_validation["test_token_label_len_ok"]
        and ate_validation["train_bio_tags_valid"] and ate_validation["val_bio_tags_valid"] and ate_validation["test_bio_tags_valid"]
        and ate_validation["train_no_illegal_i_without_b"] and ate_validation["val_no_illegal_i_without_b"] and ate_validation["test_no_illegal_i_without_b"]
        and ate_validation["train_num_records_all_o_but_has_matched_aspect"] == 0
        and ate_validation["val_num_records_all_o_but_has_matched_aspect"] == 0
        and ate_validation["test_num_records_all_o_but_has_matched_aspect"] == 0
        and ate_validation["train_num_zero_coverage_aspects"] == 0
        and ate_validation["val_num_zero_coverage_aspects"] == 0
        and ate_validation["test_num_zero_coverage_aspects"] == 0
        and ate_validation["train_num_partial_coverage_aspects"] == 0
        and ate_validation["val_num_partial_coverage_aspects"] == 0
        and ate_validation["test_num_partial_coverage_aspects"] == 0
    )),
    ("ASC validation passed", (
        asc_validation["train_all_labels_valid"] and asc_validation["val_all_labels_valid"] and asc_validation["test_all_labels_valid"]
        and asc_validation["train_records_with_bad_span"] == 0
        and asc_validation["val_records_with_bad_span"] == 0
        and asc_validation["test_records_with_bad_span"] == 0
        and asc_validation["train_records_with_bad_template"] == 0
        and asc_validation["val_records_with_bad_template"] == 0
        and asc_validation["test_records_with_bad_template"] == 0
        and asc_validation["train_records_with_invalid_match_type"] == 0
        and asc_validation["val_records_with_invalid_match_type"] == 0
        and asc_validation["test_records_with_invalid_match_type"] == 0
        and asc_validation["train_records_with_invalid_sentiment"] == 0
        and asc_validation["val_records_with_invalid_sentiment"] == 0
        and asc_validation["test_records_with_invalid_sentiment"] == 0
    )),
    ("Split manifest exported", SPLIT_MANIFEST_PATH.exists()),
]

df_checklist = pd.DataFrame(checklist, columns=["check_item", "passed"])
df_checklist.to_csv(PHASE2_CHECKLIST_PATH, index=False)

display(df_checklist)

all_passed = df_checklist["passed"].all()
print("PHASE 2 COMPLETE:", all_passed)
print("ATE strict validation summary:")
print("num_zero_coverage_aspects:", ate_validation["train_num_zero_coverage_aspects"] + ate_validation["val_num_zero_coverage_aspects"] + ate_validation["test_num_zero_coverage_aspects"])
print("num_partial_coverage_aspects:", ate_validation["train_num_partial_coverage_aspects"] + ate_validation["val_num_partial_coverage_aspects"] + ate_validation["test_num_partial_coverage_aspects"])
print("num_records_all_o_but_has_matched_aspect:", ate_validation["train_num_records_all_o_but_has_matched_aspect"] + ate_validation["val_num_records_all_o_but_has_matched_aspect"] + ate_validation["test_num_records_all_o_but_has_matched_aspect"])
print("num_records_bio_span_count_mismatch:", ate_validation["train_num_records_bio_span_count_mismatch"] + ate_validation["val_num_records_bio_span_count_mismatch"] + ate_validation["test_num_records_bio_span_count_mismatch"])
print("ASC strict validation summary:")
print("records_with_bad_span:", asc_validation["train_records_with_bad_span"] + asc_validation["val_records_with_bad_span"] + asc_validation["test_records_with_bad_span"])
print("records_with_bad_template:", asc_validation["train_records_with_bad_template"] + asc_validation["val_records_with_bad_template"] + asc_validation["test_records_with_bad_template"])
print("records_with_invalid_match_type:", asc_validation["train_records_with_invalid_match_type"] + asc_validation["val_records_with_invalid_match_type"] + asc_validation["test_records_with_invalid_match_type"])
print("records_with_invalid_sentiment:", asc_validation["train_records_with_invalid_sentiment"] + asc_validation["val_records_with_invalid_sentiment"] + asc_validation["test_records_with_invalid_sentiment"])


,check_item,passed
0,Canonical input loaded,True
1,Review holdout created,True
2,All model-ready rows assigned split,True
3,No train/val group leakage,True
4,No train/test group leakage,True
5,No val/test group leakage,True
6,Train canonical exported,True
7,Val canonical exported,True
8,Test canonical exported,True
9,ATE train exported,True


PHASE 2 COMPLETE: True
ATE strict validation summary:
num_zero_coverage_aspects: 0
num_partial_coverage_aspects: 0
num_records_all_o_but_has_matched_aspect: 0
num_records_bio_span_count_mismatch: 0
ASC strict validation summary:
records_with_bad_span: 0
records_with_bad_template: 0
records_with_invalid_match_type: 0
records_with_invalid_sentiment: 0


## Phase 2 conclusion

This phase is considered successful if the canonical finance ABSA dataset has been converted into:

1. a duplicate-aware, reproducible train/val/test split;
2. split-specific canonical files;
3. ATE-ready word-level BIO supervision files;
4. ASC-ready aspect-level classification files;
5. a review holdout for unresolved aspect-grounding cases.

The next phase will focus on model-facing preprocessing:

- tokenizer-specific conversion for DeBERTa,
- subword alignment for BIO labels,
- PyTorch/HuggingFace Dataset and DataLoader construction,
- and shared dataset interfaces for Member 3 (ATE), Member 4 (ASC), and Member 5 (joint model).